# Lasso and Ridge Regression (STP 470/Fall 2025)

### Name: *Bhavya Patel*

## General Instructions and Restrictions

This project will use the College dataset from ISLP/ISLR. A dictionary for this dataset can be found at https://islp.readthedocs.io/en/latest/datasets/College.html

Please follow the instructions and restrictions below:
- use ```statsmodels.api.datasets.get_rdataset``` to load the dataset. You already have in Canvas a notebook that uses the same function to load a different dataset.
- **Do NOT use the ISLP package** in this notebook. You should be able to construct design matrices and other necessary objects without the use of the ISLP package.

## Conceptual part (30 pts)

In the next Markdown cell you are asked to present a brief summary of both Ridge and Lasso regression. Please remove the text "Your markdown here!" and replace it with your own summary of the two methods. Your summary should provide:
- the minimization problem solved by each method;
- the similarities and differences between the two methods.

When writing this part it is recommended that you use LaTex for mathematical formulas. It is also recommended that you use Section 6.2 from the ISLP book as your main reference.

A couple of other questions that you want to answer in this markdown cell and in your final project presentation:
- when do these shrinkage methods improve over ordinary least squares?
- when is it preferable to use Ridge? What about Lasso?

Ridge and Lasso regression are shrinkage methods that modify Ordinary Least Squares (OLS) by adding a penalty to the minimization objective. This penalty shrinks coefficient estimates towards zero, introducing a small amount of bias to significantly reduce variance and improve prediction accuracy. <br>

The Minimization Problems<br>
Ridge Regression minimizes the residual sum of squares plus an $\ell_2$ penalty (squared sum of coefficients). This penalty shrinks coefficients asymptotically towards zero but never reaches it perfectly.<br>

$$\text{minimize} \sum_{i=1}^{n} \left( y_i - \beta_0 - \sum_{j=1}^{p} \beta_j x_{ij} \right)^2 + \lambda \sum_{j=1}^{p} \beta_j^2$$

Lasso Regression minimizes the residual sum of squares plus an $\ell_1$ penalty (sum of absolute values). This distinct geometry allows the penalty to force some coefficients to exactly zero, effectively removing them from the model. <br>

$$\text{minimize} \sum_{i=1}^{n} \left( y_i - \beta_0 - \sum_{j=1}^{p} \beta_j x_{ij} \right)^2 + \lambda \sum_{j=1}^{p} |\beta_j|$$

Both methods address the same fundamental issue: they sacrifice unbiasedness to lower the variance of estimates, which is crucial when predictors are highly correlated or the sample size is small relative to the number of features. Both also require standardized variables (mean 0, variance 1) because their penalties are sensitive to the scale of the predictors. The key difference lies in variable selection. Ridge regression keeps all predictors in the final model, shrinking their impact but never discarding them.  Lasso, due to its diamond-shaped constraint region, can shrink coefficients all the way to zero. This makes Lasso a feature selection tool that produces simpler, interpretable models, whereas Ridge retains the complexity of the full variable set. <br>

Shrinkage methods improve over Ordinary Least Squares when the OLS estimates suffer from high variance, typically due to multicollinearity (highly correlated predictors) or high dimensionality (when the number of predictors is close to or exceeds the sample size). In these scenarios, OLS tends to overfit the noise in the training data; shrinkage stabilizes the coefficients, preventing extreme values and reducing the overall test error.<br>

The choice between Ridge and Lasso depends on the sparsity of the underlying data. Ridge Regression is preferable when the relationship is "dense," meaning you believe most or all predictors contribute small but non-zero effects to the response. Lasso Regression is the better choice when the relationship is "sparse," meaning you suspect only a small subset of the predictors are actually relevant. Lasso will correctly zero out the irrelevant noise variables, offering a cleaner and more interpretable model.<br>

## Python Code. Section 1 (10 pts)

Use as many code cells as necessary below to do the following:
1) import necessary packages;

2) read the "College" dataset as instructed above;

3) perform a brief  Exploratory Data Analysis (EDA) that will:

  - print the names of all columns in the dataset and how many non-null instances there are in each column;
    
  - determine all columns that are categorical, i.e. ```category``` type;
    
  - print some simple statistics for numerical variables, such as mean, std, min, max, etc.

4) use ```train_test_split``` to split the data into a training set and a test set (when reporting your results, please make sure that you include the case with a 70%-30% train-test split and a ```random_state = 0```

In [1]:
import statsmodels.api as sm
dataset = sm.datasets.get_rdataset("College", "ISLR")
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error


College = dataset.data
College.columns

Index(['Private', 'Apps', 'Accept', 'Enroll', 'Top10perc', 'Top25perc',
       'F.Undergrad', 'P.Undergrad', 'Outstate', 'Room.Board', 'Books',
       'Personal', 'PhD', 'Terminal', 'S.F.Ratio', 'perc.alumni', 'Expend',
       'Grad.Rate'],
      dtype='object')

In [2]:
College.info()
categorical_cols = College.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"\nCategorical Columns: {categorical_cols}\n")
print(College.describe())

train_set, test_set = train_test_split(College, test_size=0.3, random_state=0)
print(f"\n\nTraining set shape: {train_set.shape}")
print(f"Test set shape: {test_set.shape}")


<class 'pandas.core.frame.DataFrame'>
Index: 777 entries, Abilene Christian University to York College of Pennsylvania
Data columns (total 18 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Private      777 non-null    object 
 1   Apps         777 non-null    int64  
 2   Accept       777 non-null    int64  
 3   Enroll       777 non-null    int64  
 4   Top10perc    777 non-null    int64  
 5   Top25perc    777 non-null    int64  
 6   F.Undergrad  777 non-null    int64  
 7   P.Undergrad  777 non-null    int64  
 8   Outstate     777 non-null    int64  
 9   Room.Board   777 non-null    int64  
 10  Books        777 non-null    int64  
 11  Personal     777 non-null    int64  
 12  PhD          777 non-null    int64  
 13  Terminal     777 non-null    int64  
 14  S.F.Ratio    777 non-null    float64
 15  perc.alumni  777 non-null    int64  
 16  Expend       777 non-null    int64  
 17  Grad.Rate    777 non-null    int64  
dtypes: 

## Python Code. Section 2 (10 pts)

We want to predict the number of applications received using the other **numerical** variables in the College dataset.

**Question 1.** Fit a linear regression model using ordinary least squares on the training set.

In [3]:

X_train = train_set.drop(columns=['Apps', 'Private'])
y_train = train_set['Apps']

linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

print(f"Intercept: {linear_model.intercept_:.2f}")
print("Coefficients:", linear_model.coef_)

Intercept: -898.82
Coefficients: [ 1.28901865e+00 -4.29186910e-01  5.33794144e+01 -1.70153958e+01
  1.14763773e-01  5.04659304e-02 -7.84506032e-02  1.89953300e-01
  8.55351934e-03 -4.39212378e-02 -7.18595741e+00 -1.38946565e+00
  1.49695709e+01 -8.82562339e+00  6.99794434e-02  9.57809580e+00]


## Python Code. Section 3 (15 pts)

We want to predict the number of applications received using the other **numerical** variables in the College dataset.

**Question 2.** Fit a **Ridge regression model** with $\lambda$ chosen by cross validation (use $K=10$ folds when reporting the results).

**Remark.** Section 6.5.2 in the ISLP notebook presents a Python Lab showing how to fit  both Ridge and Lasso.

In [4]:

alphas = 10**np.linspace(4, -2, 100)

ridge_model = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge_cv', RidgeCV(alphas=alphas, cv=10, scoring='neg_mean_squared_error'))
])

ridge_model.fit(X_train, y_train)

ridge_alpha = ridge_model.named_steps['ridge_cv'].alpha_
print(f"Best 𝜆 chosen by 10-fold CV: {ridge_alpha:.4f}")

Best 𝜆 chosen by 10-fold CV: 1.7475


## Python Code. Section 4 (15 pts)

We want to predict the number of applications received using the other **numerical** variables in the College dataset.

**Question 3.** Fit a **Lasso regression model** with $\lambda$ chosen by cross validation (use $K=10$ folds when reporting the results).

**Remark.** Section 6.5.2 in the ISLP notebook presents a Python Lab showing how to fit  both Ridge and Lasso.

In [5]:

lasso_model = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso_cv', LassoCV(cv=10, random_state=0, max_iter=10000))
])

lasso_model.fit(X_train, y_train)
lasso_alpha = lasso_model.named_steps['lasso_cv'].alpha_
print(f"Best 𝜆 chosen by 10-fold CV: {lasso_alpha:.4f}\n")

lasso_coefs = lasso_model.named_steps['lasso_cv'].coef_
feature_names = X_train.columns

coef_dict = dict(zip(feature_names, lasso_coefs))
for feature, coef in coef_dict.items():
    if coef == 0:
        print(f"{feature}: 0.00 (Dropped)")
    else:
        print(f"{feature}: {coef:.4f}")

print(f"\nNumber of non-zero features: {np.sum(lasso_coefs != 0)}")

Best 𝜆 chosen by 10-fold CV: 11.3596

Accept: 2956.8901
Enroll: 0.00 (Dropped)
Top10perc: 778.3442
Top25perc: -201.7927
F.Undergrad: 324.8176
P.Undergrad: 74.3055
Outstate: -240.6420
Room.Board: 191.7697
Books: 0.00 (Dropped)
Personal: -13.0118
PhD: -88.7597
Terminal: -18.8458
S.F.Ratio: 38.4809
perc.alumni: -110.3585
Expend: 303.0851
Grad.Rate: 124.5461

Number of non-zero features: 14


## Reporting (20 pts)

Use as many markdown and code cells as necessary to report your findings by addressing the issues below:

1) For **Question 1**, report the test error obtained (Use a 70%-30% train-test split and a ```random_state = 0```).

2) For **Question 2**, report the value of $\lambda$ obtained by cross-validation and the test error (Use a 70%-30% train-test split, a ```random_state = 0``` and a $10$-fold cross-validation).

3) For **Question 2**, report the value of $\lambda$ obtained by cross-validation, the test error and the number of non-zero coefficient estimates (Use a 70%-30% train-test split, a ```random_state = 0``` and a $10$-fold cross-validation).

In [6]:

X_test = test_set.drop(columns=['Apps', 'Private'])
y_test = test_set['Apps']

ols_pred = linear_model.predict(X_test)
ols_mse = mean_squared_error(y_test, ols_pred)

ridge_lambda = ridge_model.named_steps['ridge_cv'].alpha_
ridge_pred = ridge_model.predict(X_test)
ridge_mse = mean_squared_error(y_test, ridge_pred)

lasso_model = lasso_model.named_steps['lasso_cv']
lasso_lambda = lasso_model.alpha_
lasso_pred = lasso_model.predict(X_test)
lasso_mse = mean_squared_error(y_test, lasso_pred)
non_zero_cnt = np.sum(lasso_model.coef_ != 0)

print("OLS Linear Regression:")
print(f"Test MSE:   {ols_mse:,.2f}")
print("\nRidge Regression:")
print(f"Lambda:   {ridge_lambda:.4f}")
print(f"Test MSE: {ridge_mse:,.2f}")
print("\nLasso Regression:")
print(f"Lambda:   {lasso_lambda:.4f}")
print(f"Test MSE: {lasso_mse:,.2f}")
print(f"Features: {non_zero_cnt} (out of {X_test.shape[1]})")

OLS Linear Regression:
Test MSE:   1,598,872.11

Ridge Regression:
Lambda:   1.7475
Test MSE: 1,676,593.82

Lasso Regression:
Lambda:   11.3596
Test MSE: 146,831,708,086,063.59
Features: 14 (out of 16)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LassoCV was fitted without feature names
  warnings.warn(


## Interpretation
#### <b>OLS Linear Regression</b>
Test Error MSE:   1,598,872.11 <br>
This is the baseline performance for the unregularized model. An MSE of ~1.6 million corresponds to a RMSE of approximately 1,264 applications. This means that, on average, the OLS model's predictions are off by about 1,264 applications. We use this value to judge if the regularization in the next steps improves accuracy.
<br>

#### <b>Ridge Regression</b>
$\lambda$:   1.7475<br>
Test Error MSE: 1,676,593.82<br>
Cross-validation selected a small penalty ($\lambda \approx 1.75$). However, the Ridge Test MSE is slightly higher than the OLS baseline (1.67M vs 1.60M). This suggests that for this specific train-test split, the unregularized OLS model was already quite efficient. The bias introduced by the Ridge penalty slightly outweighed the variance reduction, leading to a marginal decrease in performance on the test set.

#### <b>Lasso Regression</b>
$\lambda$: 11.3596<br>
Test Error MSE: 146,831,708,086,063.59<br>
Non-zero Coefficients: 14 (out of 16)<br>
The Lasso model performed significantly worse than both OLS and Ridge models. It retained 14 of the 16 variables, dropping only 2 features. The MSE of ~146 trillion is disproportionately large. This indicates the model likely failed to converge properly or encountered a numerical stability issue during the fitting process. In a typical scenario, we would expect the Lasso MSE to be comparable to the OLS/Ridge results (around 1-2 million).
<br><br>
Based on these specific numbers, the OLS model is the best performing model.